# PETase figure generation
access precalculated data from huggingface: https://huggingface.co/datasets/fmoorhof/selectzyme-app-data/tree/main/petase
make multifigure plot with the plotting objectives: query_term, PET, domain 

In [ ]:
#!pip install --quiet huggingface-hub==1.2.3
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

In [ ]:
def import_results(dataset_name: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """
    Imports and loads results from a dataset downloaded from Hugging Face Hub.
    Args:
        dataset_name (str): Name of the dataset to fetch from Hugging Face Hub.
    Returns:
        tuple: A tuple containing the following:
            - pd.DataFrame: DataFrame loaded from "df.parquet".
            - np.ndarray: Reduced feature matrix loaded from "x_red_mst_slc.npz".
            - np.ndarray: Minimum spanning tree (MST) loaded from "x_red_mst_slc.npz".
            - np.ndarray: Linkage matrix loaded from "x_red_mst_slc.npz".
    """
    # Download files from Hugging Face Hub
    df_path = hf_hub_download(repo_id="fmoorhof/selectzyme-app-data", 
                              filename=f"{dataset_name}/df.parquet", 
                              repo_type="dataset")
    npz_path = hf_hub_download(repo_id="fmoorhof/selectzyme-app-data", 
                               filename=f"{dataset_name}/x_red_mst_slc.npz", 
                               repo_type="dataset")

    # Load data
    df = pd.read_parquet(df_path)
    adata = np.load(npz_path)
    X_red = adata["X_red"]
    mst = adata["mst"]
    linkage = adata["linkage"]

    return df, X_red, mst, linkage

In [ ]:
df, X_red, mst, linkage = import_results("petase")
df

In [ ]:
# plotting instructions

def _ensure_xy_numeric(df):
    df = df.copy()
    for c in ["x", "y"]:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.replace(",", ".", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _collapse_to_allowed_categories(
    s: pd.Series,
    allowed: list[str],
    other_label: str = "Others",
    na_label: str = "NA",
):
    """Collapse everything not in `allowed` into `other_label`."""
    s2 = s.astype(str).fillna(na_label)
    allowed_str = [str(a) for a in allowed]
    allowed_present = [a for a in allowed_str if (s2 == a).any()]

    s_collapsed = s2.where(s2.isin(set(allowed_str)), other_label)
    categories = allowed_present + ([other_label] if (s_collapsed == other_label).any() else [])
    return pd.Categorical(s_collapsed, categories=categories, ordered=False)

def _get_nature_cmap(K: int):
    """Muted, publication-style categorical palette (ColorBrewer via Matplotlib)."""
    if K <= 8:
        return plt.get_cmap("Accent", K)
    if K <= 12:
        return plt.get_cmap("Accent", K)
    return plt.get_cmap("tab20c", K)

def _scatter_panel_categories(
    ax,
    df,
    col,
    *,
    x_col: str = "x",
    y_col: str = "y",
    point_size: float = 6,
    alpha: float = 0.75,
    allowed: list[str] | None = None,
    other_label: str = "Others",
    minority_on_top: bool = True,
):
    """Scatter UMAP colored by `col` with optional collapsing and minority-on-top draw order."""
    if col not in df.columns:
        ax.set_title(f"{col} (missing)")
        ax.axis("off")
        return

    s = df[col]
    if allowed is not None:
        s_plot = _collapse_to_allowed_categories(
            s,
            allowed=allowed,
            other_label=other_label,
        )
        labels = list(s_plot.categories)
    else:
        # No collapsing: order legend by frequency (most abundant first).
        s2 = s.astype(str).fillna("NA")
        vc = s2.value_counts(dropna=False)
        labels = vc.index.tolist()
        s_plot = pd.Categorical(s2, categories=labels, ordered=False)

    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()

    # Integer codes 0..K-1 aligned to `labels`
    codes = pd.Series(s_plot).map({lab: i for i, lab in enumerate(labels)}).to_numpy()
    if np.any(pd.isna(codes)):
        codes = np.nan_to_num(codes, nan=0).astype(int)
    else:
        codes = codes.astype(int)

    # Draw order: plot majority first, minority last (on top).
    if minority_on_top and len(labels) > 1:
        s_vals = pd.Series(s_plot).astype(str)
        vc_plot = s_vals.value_counts(dropna=False)
        # higher frequency should be drawn earlier -> descending frequency
        freq_per_point = s_vals.map(vc_plot).to_numpy()
        order = np.argsort(-freq_per_point, kind="mergesort")
        x = x[order]
        y = y[order]
        codes = codes[order]

    K = max(len(labels), 1)
    cmap = _get_nature_cmap(K)
    norm = BoundaryNorm(np.arange(-0.5, K + 0.5, 1), K)

    ax.scatter(
        x,
        y,
        c=codes,
        cmap=cmap,
        norm=norm,
        s=point_size,
        alpha=alpha,
        linewidths=0,
    )

    # ax.set_title(col)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.grid(True, alpha=0.2)

    # Legend colors must match the discrete LUT exactly
    handles = []
    for i, lab in enumerate(labels):
        color = cmap.colors[i] if hasattr(cmap, "colors") else cmap(i)
        handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="o",
                color="none",
                markerfacecolor=color,
                markersize=6,
                alpha=alpha,
                label=lab,
            )
        )
    ax.legend(
        handles=handles,
        title=col,
        loc="upper right",
        bbox_to_anchor=(0.5, -0.15),
        ncol=6,
        frameon=False,
        fontsize=10,
        title_fontsize=12,
    )

In [ ]:
# plotting
assert len(df) == X_red.shape[0], "df and X_red must have the same number of rows"

plot_df = df[["accession", "query_term", "PET", "domain"]].copy()
plot_df["x"] = X_red[:, 0]
plot_df["y"] = X_red[:, 1]

# Plot styling (muted publication palette + minority-on-top)
point_size = 6
alpha_val = 0.75

plot_df2 = _ensure_xy_numeric(plot_df).dropna(subset=["x", "y"]).copy()

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

# Only show these query_term labels; everything else -> 'Others'
query_term_allowed = ["IPR003140", "IPR013818", "PF01083", "cd00312"]  # , "ec:3.1.1.3"

_scatter_panel_categories(
    axes[0],
    plot_df2,
    "query_term",
    point_size=point_size,
    alpha=alpha_val,
    allowed=query_term_allowed,
    other_label="Others",
    minority_on_top=True,
 )

# PET and domain: no special collapsing; still use minority_on_top to avoid "unknown" blocking
_scatter_panel_categories(axes[1], plot_df2, "PET", point_size=point_size, alpha=alpha_val, minority_on_top=True)
_scatter_panel_categories(axes[2], plot_df2, "domain", point_size=point_size, alpha=alpha_val, minority_on_top=True)

plt.tight_layout()
plt.savefig("petase_plot.png", dpi=600)
# plt.savefig("petase_plot.svg")
plt.show()
